In [1]:
import pandas  as pd
import numpy as np
import pdb, os, datetime, itertools, time, hashlib
import pyprojroot, sys
from pyprojroot.criterion import has_file
sys.path.insert(0, str(pyprojroot.find_root(has_file("pyproject.toml"))))
from dotenv import load_dotenv

load_dotenv()
from lib.flp001 import *

/workspace/worker/pj/Chrono/genuis/mizar
/workspace/worker/pj/Chrono/genuis/mizar/config/contract.toml


In [2]:
method = 'ricso2'
task_id = '113001'
session = '20260507'
period = 5
category = 1
left_instruments = 'rbb'
right_instruments = 'hcb'

In [3]:
results = fetch_data(method=method, task_id=task_id, instruments=left_instruments, 
          period=period, session=session, category=category)
results['name'] = results['name'].astype(int).astype(str)
results['ic_mean'] = pd.to_numeric(results['ic_mean'], errors='coerce')
results['total_ic'] = pd.to_numeric(results['total_ic'], errors='coerce')
results['abs_ic_mean'] = results['ic_mean'].abs()
results['abs_total_ic'] = results['total_ic'].abs()

/workspace/worker/pj/Chrono/genuis/mizar/records/ricso2/rbb/rulex/113001/nxt1_ret_5h/20260507


In [4]:
left_results = results[results['instrument'] == left_instruments]
right_results = results[results['instrument'] == right_instruments]

In [5]:
left_results[['instrument','name', 'expression', 'avg_ret', 'total_ret', 
              'max_dd', 'calmar', 'win_rate', 'pl_ratio', 'ic_mean', 'icir',
              'sharpe', 'ann_sharpe','total_ic']].head()

,instrument,name,expression,avg_ret,total_ret,max_dd,calmar,win_rate,pl_ratio,ic_mean,icir,sharpe,ann_sharpe,total_ic
0,rbb,10000983,"MDIFF(90,MDIFF(90,'iv004_2_1_2_3_1'))",0.16,2053.44,-9.31,2.78,None,None,-0.0472,-0.1765,0.02,3.09,-0.0247
2,rbb,10030178,"EMA(90,'tc005_2_2_3_0')",0.19,3836.32,-16.84,1.88,None,None,-0.0866,-0.3309,0.02,2.90,-0.0227
4,rbb,10037316,"MRANK(60,'tc004_1_1_2_0')",0.13,1131.58,-10.35,2.00,None,None,-0.0447,-0.1676,0.02,2.69,-0.0211
6,rbb,10046299,"MADiff(90,MADiff(90,MADiff(90,'ixy002_2_3_0')))",0.14,1376.65,-11.74,1.91,None,None,0.0362,0.1349,0.02,2.64,0.0206
8,rbb,10047868,"MDIFF(60,MIR(10,'tc012_1_1_2_1'))",0.20,5559.60,-10.61,3.33,None,None,-0.0508,-0.1929,0.03,3.42,-0.0266


In [6]:
right_results[['instrument','name', 'expression', 'avg_ret', 'total_ret', 
              'max_dd', 'calmar', 'win_rate', 'pl_ratio', 'ic_mean', 'icir',
              'sharpe', 'ann_sharpe','total_ic']].head()

,instrument,name,expression,avg_ret,total_ret,max_dd,calmar,win_rate,pl_ratio,ic_mean,icir,sharpe,ann_sharpe,total_ic
1,hcb,10000983,"MDIFF(90,MDIFF(90,'iv004_2_1_2_3_1'))",0.11,597.29,-21.92,0.87,None,None,-0.0383,-0.1417,0.02,2.08,-0.0162
3,hcb,10030178,"EMA(90,'tc005_2_2_3_0')",0.13,935.71,-21.73,1.08,None,None,-0.0786,-0.296,0.01,2.01,-0.0154
5,hcb,10037316,"MRANK(60,'tc004_1_1_2_0')",0.10,410.98,-19.97,0.79,None,None,-0.0376,-0.14,0.01,1.84,-0.0146
7,hcb,10046299,"MADiff(90,MADiff(90,MADiff(90,'ixy002_2_3_0')))",0.12,738.77,-22.55,0.94,None,None,0.0340,0.1271,0.02,2.20,0.0172
9,hcb,10047868,"MDIFF(60,MIR(10,'tc012_1_1_2_1'))",0.15,1360.73,-16.18,1.69,None,None,-0.0415,-0.1568,0.02,2.39,-0.0184


In [7]:
### 初步筛选双品种保持一致

In [8]:
ic_mean = 0.03
total_ic = 0.015
ann_sharpe = 2
calmar = 1.5

In [9]:
left_results1 = left_results[(left_results['abs_ic_mean'] >= ic_mean) & (
    left_results['abs_total_ic'] >= total_ic
) & (
    left_results['ann_sharpe'] >= ann_sharpe
) & (
    left_results['calmar'] >= calmar
)]

left_results1.head()

,name,expression,avg_ret,total_ret,sharpe,ann_sharpe,max_dd,calmar,win_rate,pl_ratio,...,factor_ac,ret_ac,roll_win,resampling_win,holding_profit,total_ic,category,instrument,abs_ic_mean,abs_total_ic
0,10000983,"MDIFF(90,MDIFF(90,'iv004_2_1_2_3_1'))",0.16,2053.44,0.02,3.09,-9.31,2.78,None,None,...,0.0973,-0.0377,15.0,5.0,nxt1_ret_5h,-0.0247,p,rbb,0.0472,0.0247
2,10030178,"EMA(90,'tc005_2_2_3_0')",0.19,3836.32,0.02,2.90,-16.84,1.88,None,None,...,0.9137,-0.0593,15.0,5.0,nxt1_ret_5h,-0.0227,p,rbb,0.0866,0.0227
4,10037316,"MRANK(60,'tc004_1_1_2_0')",0.13,1131.58,0.02,2.69,-10.35,2.00,None,None,...,0.0040,-0.0720,15.0,5.0,nxt1_ret_5h,-0.0211,p,rbb,0.0447,0.0211
6,10046299,"MADiff(90,MADiff(90,MADiff(90,'ixy002_2_3_0')))",0.14,1376.65,0.02,2.64,-11.74,1.91,None,None,...,0.0282,-0.0593,15.0,5.0,nxt1_ret_5h,0.0206,p,rbb,0.0362,0.0206
8,10047868,"MDIFF(60,MIR(10,'tc012_1_1_2_1'))",0.20,5559.60,0.03,3.42,-10.61,3.33,None,None,...,0.4925,-0.0381,15.0,5.0,nxt1_ret_5h,-0.0266,p,rbb,0.0508,0.0266


In [10]:
right_results1 = right_results[(right_results['abs_ic_mean'] >= ic_mean) & (
    right_results['abs_total_ic'] >= total_ic
) & (
    right_results['ann_sharpe'] >= ann_sharpe
) & (
    right_results['calmar'] >= calmar
)]

right_results1.head()

,name,expression,avg_ret,total_ret,sharpe,ann_sharpe,max_dd,calmar,win_rate,pl_ratio,...,factor_ac,ret_ac,roll_win,resampling_win,holding_profit,total_ic,category,instrument,abs_ic_mean,abs_total_ic
9,10047868,"MDIFF(60,MIR(10,'tc012_1_1_2_1'))",0.15,1360.73,0.02,2.39,-16.18,1.69,None,None,...,0.5022,0.0116,15.0,5.0,nxt1_ret_5h,-0.0184,p,hcb,0.0415,0.0184
87,10383399,"MDPO(60,WMA(60,'tn004_1_2_1'))",0.16,1541.47,0.02,2.42,-16.51,1.74,None,None,...,0.9857,0.0121,15.0,5.0,nxt1_ret_5h,-0.0183,p,hcb,0.0453,0.0183
113,10514607,"MPERCENT(90,MPERCENT(90,MSUM(5,'iv012_1_2_0')))",0.16,1576.16,0.02,2.74,-17.00,1.70,None,None,...,0.0887,0.0022,15.0,5.0,nxt1_ret_5h,0.0218,p,hcb,0.0343,0.0218
131,10566435,"MIR(10,'tc012_1_1_2_1')",0.15,1360.73,0.02,2.39,-16.18,1.69,None,None,...,0.5022,0.0116,15.0,5.0,nxt1_ret_5h,-0.0184,p,hcb,0.0415,0.0184
195,10935309,"MMaxDiff(90,MPERCENT(60,'ixy004_1_2_1'))",0.13,828.69,0.02,2.36,-11.23,1.98,None,None,...,0.0027,0.0071,15.0,5.0,nxt1_ret_5h,-0.0184,p,hcb,0.0356,0.0184


In [11]:
# 关注的核心绩效指标
perf_cols = ['ann_sharpe', 'calmar', 'abs_ic_mean', 'abs_total_ic', 'max_dd', 'avg_ret', 'total_ic', 'ic_mean']

# 挑选需要的列参与合并
key_cols = ['name', 'expression']
merged_results = pd.merge(
    left_results1[key_cols + perf_cols],
    right_results1[key_cols + perf_cols],
    on=key_cols,
    how='inner',
    suffixes=('_{0}'.format(left_instruments), '_{0}'.format(right_instruments))
)

# 按照 [name, expression, ann_sharpe_hcb, ann_sharpe_rb, ...] 的顺序排布便于直接对比
display_cols = key_cols + [f"{col}_{suf}" for col in perf_cols for suf in [left_instruments, right_instruments]]
merged_results = merged_results[display_cols]

display(merged_results.sort_values(by=["ann_sharpe_{0}".format(left_instruments)]))

,name,expression,ann_sharpe_rbb,ann_sharpe_hcb,calmar_rbb,calmar_hcb,abs_ic_mean_rbb,abs_ic_mean_hcb,abs_total_ic_rbb,abs_total_ic_hcb,max_dd_rbb,max_dd_hcb,avg_ret_rbb,avg_ret_hcb,total_ic_rbb,total_ic_hcb,ic_mean_rbb,ic_mean_hcb
4,10935309,"MMaxDiff(90,MPERCENT(60,'ixy004_1_2_1'))",2.72,2.36,1.52,1.98,0.0408,0.0356,0.0229,0.0184,-16.15,-11.23,0.15,0.13,-0.0229,-0.0184,-0.0408,-0.0356
1,10383399,"MDPO(60,WMA(60,'tn004_1_2_1'))",3.29,2.42,2.40,1.74,0.0542,0.0453,0.0257,0.0183,-15.24,-16.51,0.21,0.16,-0.0257,-0.0183,-0.0542,-0.0453
2,10514607,"MPERCENT(90,MPERCENT(90,MSUM(5,'iv012_1_2_0')))",3.41,2.74,2.79,1.70,0.0417,0.0343,0.0263,0.0218,-11.06,-17.00,0.19,0.16,0.0263,0.0218,0.0417,0.0343
0,10047868,"MDIFF(60,MIR(10,'tc012_1_1_2_1'))",3.42,2.39,3.33,1.69,0.0508,0.0415,0.0266,0.0184,-10.61,-16.18,0.20,0.15,-0.0266,-0.0184,-0.0508,-0.0415
3,10566435,"MIR(10,'tc012_1_1_2_1')",3.42,2.39,3.33,1.69,0.0508,0.0415,0.0266,0.0184,-10.61,-16.18,0.20,0.15,-0.0266,-0.0184,-0.0508,-0.0415
5,10964248,"MIR(10,MDIFF(60,'tc012_1_1_2_1'))",3.42,2.39,3.33,1.69,0.0508,0.0415,0.0266,0.0184,-10.61,-16.18,0.20,0.15,-0.0266,-0.0184,-0.0508,-0.0415


In [12]:
import numpy as np

# 1. 指定判断方向所用的列（使用 left_instruments 即 hcb 的列）
total_ic_col = f'total_ic_{left_instruments}'  # 即 'total_ic_hcb'
ic_mean_col = f'ic_mean_{left_instruments}'    # 即 'ic_mean_hcb'

# 2. 计算方向字段：都为正为 1，都为负为 -1，不一致为 0
conditions = [
    (merged_results[total_ic_col] > 0) & (merged_results[ic_mean_col] > 0),
    (merged_results[total_ic_col] < 0) & (merged_results[ic_mean_col] < 0)
]
choices = [1, -1]
merged_results['direction'] = np.select(conditions, choices, default=0)

# 3. 构造 plot 路径（已修正多余的 '+'，并将 results['name'] 改为 merged_results['name']）
session_name = "/{0}".format(session) if category == 1 else "/d{0}".format(session)  
merged_results['plot'] = (
    base_path + '/' + str(method) + '/' + left_instruments \
    + '/rulex' + '/' + str(task_id) + f"/nxt1_ret_{period}h" \
    + session_name + '/plot/' + merged_results['name'].astype(str) + '.png'
)

# 4. 保留指定列并重命名，渲染可点击链接
merged_results = merged_results[['expression', 'plot', 'name', 'direction']].rename(
    columns={'name': 'factor_id', 'expression': 'formula'}
)
merged_results['plot'] = merged_results['plot'].apply(make_clickable)

#display(merged_results.)

In [13]:
to_html(merged_results)

url,formula,factor_id,direction,plot
10047868,"MDIFF(60,MIR(10,'tc012_1_1_2_1'))",10047868,-1,/workspace/worker/pj/Chrono/genuis/mizar/records/ricso2/rbb/rulex/113001/nxt1_ret_5h/20260507/plot/10047868.png
10383399,"MDPO(60,WMA(60,'tn004_1_2_1'))",10383399,-1,/workspace/worker/pj/Chrono/genuis/mizar/records/ricso2/rbb/rulex/113001/nxt1_ret_5h/20260507/plot/10383399.png
10514607,"MPERCENT(90,MPERCENT(90,MSUM(5,'iv012_1_2_0')))",10514607,1,/workspace/worker/pj/Chrono/genuis/mizar/records/ricso2/rbb/rulex/113001/nxt1_ret_5h/20260507/plot/10514607.png
10566435,"MIR(10,'tc012_1_1_2_1')",10566435,-1,/workspace/worker/pj/Chrono/genuis/mizar/records/ricso2/rbb/rulex/113001/nxt1_ret_5h/20260507/plot/10566435.png
10935309,"MMaxDiff(90,MPERCENT(60,'ixy004_1_2_1'))",10935309,-1,/workspace/worker/pj/Chrono/genuis/mizar/records/ricso2/rbb/rulex/113001/nxt1_ret_5h/20260507/plot/10935309.png
10964248,"MIR(10,MDIFF(60,'tc012_1_1_2_1'))",10964248,-1,/workspace/worker/pj/Chrono/genuis/mizar/records/ricso2/rbb/rulex/113001/nxt1_ret_5h/20260507/plot/10964248.png


In [14]:
merged_results

,formula,plot,factor_id,direction
0,"MDIFF(60,MIR(10,'tc012_1_1_2_1'))","<a target=""_blank"" href=""/workspace/worker/pj/...",10047868,-1
1,"MDPO(60,WMA(60,'tn004_1_2_1'))","<a target=""_blank"" href=""/workspace/worker/pj/...",10383399,-1
2,"MPERCENT(90,MPERCENT(90,MSUM(5,'iv012_1_2_0')))","<a target=""_blank"" href=""/workspace/worker/pj/...",10514607,1
3,"MIR(10,'tc012_1_1_2_1')","<a target=""_blank"" href=""/workspace/worker/pj/...",10566435,-1
4,"MMaxDiff(90,MPERCENT(60,'ixy004_1_2_1'))","<a target=""_blank"" href=""/workspace/worker/pj/...",10935309,-1
5,"MIR(10,MDIFF(60,'tc012_1_1_2_1'))","<a target=""_blank"" href=""/workspace/worker/pj/...",10964248,-1


In [15]:
### 转成临时格式
merged_results1 = merged_results[['formula','direction']]
merged_results1['source'] = session
merged_results1['category'] = 'p'
merged_results1.head()

,formula,direction,source,category
0,"MDIFF(60,MIR(10,'tc012_1_1_2_1'))",-1,20260507,p
1,"MDPO(60,WMA(60,'tn004_1_2_1'))",-1,20260507,p
2,"MPERCENT(90,MPERCENT(90,MSUM(5,'iv012_1_2_0')))",1,20260507,p
3,"MIR(10,'tc012_1_1_2_1')",-1,20260507,p
4,"MMaxDiff(90,MPERCENT(60,'ixy004_1_2_1'))",-1,20260507,p


In [16]:
session_name = f'd{session}' if category == 2 else str(session)
file_path = Path(base_path) / method / left_instruments / 'rulex' / task_id / (f'nxt1_ret_{period}h')
file_path

PosixPath('/workspace/worker/pj/Chrono/genuis/mizar/records/ricso2/rbb/rulex/113001/nxt1_ret_5h')

In [17]:
merged_results1.to_csv(os.path.join(file_path, "{0}_cohort.csv".format(session)),encoding='UTF-8')